# Tutorial 2: Zero-Leakage Alignment and the Four-Step Operational TTA Protocol

This pure-CPU tutorial walks through the core engineering invariants of **SWI-TTA**:
1. Verify strict **zero-leakage input period alignment** (`prepare_aligned_input`).
2. Verify **Resolvable-Depth $V_s$ RMSE** (`vs_rmse_res`) ignores depths deeper than $z_{\text{res}}$.
3. Run `run_operational_protocol` live on CPU over synthetic stations spanning both below-floor (`r_0 < 2.0`, retained zero-shot) and above-floor (`r_0 >= 2.0`, routed to unfrozen Holdout ES + converged `disba` verification/rollback) regimes.
4. Inspect the closed-loop 14-cohort audit archive (`results_summary/operational_protocol_audit.json`).

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

from swi_tta.adaptation.safe_tta import run_operational_protocol
from swi_tta.data.alignment import prepare_aligned_input
from swi_tta.evaluation.metrics import vs_rmse_res
from swi_tta.physics.disba_referee import forward_dispersion_diag

# 1. Zero-leakage alignment check
grid_t = np.linspace(0.25, 2.0, 30, dtype=np.float32)
obs_t = np.linspace(0.30, 1.80, 20, dtype=np.float32)
ph = (0.45 + 0.45 * obs_t)[None, :]
gp = np.zeros_like(ph)
obs_mask = np.ones((1, 20, 2), dtype=np.float32)
ho_mask = np.zeros((1, 20, 2), dtype=np.float32)
ho_mask[:, [3, 8, 13, 16], :] = 1.0

ph_corrupted = ph.copy()
ph_corrupted[:, [3, 8, 13, 16]] = 999.0
x_clean, _ = prepare_aligned_input(obs_t[None, :], ph, gp, obs_mask, ho_mask, grid_t, remove_group=True)
x_corrupt, _ = prepare_aligned_input(obs_t[None, :], ph_corrupted, gp, obs_mask, ho_mask, grid_t, remove_group=True)
print('Max alignment diff after corrupting held-out periods:', float(np.max(np.abs(x_clean - x_corrupt))))

In [ ]:
# 2. Live CPU execution of run_operational_protocol on 2 synthetic shallow stations
#    Station 0: accurate zero-shot (r_0 < 2.0 -> retained as zero-shot)
#    Station 1: strongly shifted zero-shot (r_0 >= 2.0 -> routed to unfrozen Holdout ES + exact disba rollback)
depths = np.linspace(0.0, 2.76, 70, dtype=np.float64)
periods_1d = np.linspace(0.30, 1.60, 20, dtype=np.float64)
vs_true = (0.45 + 0.55 * depths).astype(np.float32)
obs_true = forward_dispersion_diag(vs_true, depths, periods_1d, dc=2.5e-5)['phase_vel'].astype(np.float32)

vs_init = np.stack([vs_true * 1.004, vs_true * 1.20], axis=0).astype(np.float32)
periods = np.stack([periods_1d, periods_1d], axis=0).astype(np.float32)
phase_vel = np.stack([obs_true, obs_true], axis=0).astype(np.float32)
group_vel = np.zeros_like(phase_vel)
mask = np.ones((2, 20, 2), dtype=np.float32)
mask[:, :, 1] = 0.0  # shallow phase-only

class _TinySurrogateMember(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('grid_dense', torch.linspace(0.20, 2.00, 32))
        self.scale = nn.Parameter(torch.tensor(0.92))

    def forward(self, vs, per_q):
        # Smooth differentiable proxy mapping mean shallow Vs to phase velocity
        mean_shallow = vs[:, :20].mean(dim=1, keepdim=True)
        ph = self.scale * mean_shallow * (0.70 + 0.30 * per_q)
        gp = torch.zeros_like(ph)
        return torch.stack([ph, gp], dim=1)

class _TinyEnsemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.members = nn.ModuleList([_TinySurrogateMember()])
    def predict_with_uncertainty(self, vs, per_q):
        pred = self.members[0](vs, per_q)
        return pred, torch.zeros_like(pred), pred.unsqueeze(0)

vs_out, info = run_operational_protocol(
    vs_init, _TinyEnsemble(), depths, periods, phase_vel, group_vel, mask, mask,
    domain='shallow', sigma_hat_d=0.01, dc_shallow=2.5e-5, num_steps=15, check_interval=5, disba_workers=1, device='cpu',
)
print('Initial r_0:', np.round(info['r0'], 2))
print('Route actions:', info['route_action'])
print('Reverted:', info['reverted'].tolist(), '| Reasons:', info['revert_reason'])
print('Station 0 bit-identical to zero-shot:', bool(np.array_equal(vs_out[0], vs_init[0])))

In [ ]:
# 3. Inspect archived 14-cohort closed-loop protocol results
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'tutorials' else Path.cwd()
audit = json.loads((REPO_ROOT / 'results_summary' / 'operational_protocol_audit.json').read_text(encoding='utf-8'))
print(f"Cohorts audited: {len(audit['cohorts'])} | Default freeze: {audit['protocol_definition']['default_switches']['enable_zres_freeze']}")
for c in audit['cohorts']:
    ac = c['all_checkpoints']
    metric = (
        f"{ac['protocol_delta_rmse_pct']:+.2f}% RMSE"
        if not c['is_field']
        else f"HO {ac['ho_median_zs']*100:.2f}% -> {ac['ho_median_protocol']*100:.2f}%"
    )
    print(f"{c['display_label']:<38s} | {metric:<24s} | routed {ac['routed_pct']:5.1f}% | disba saved {ac['disba_call_savings_pct']:+5.1f}%")